# Portugal 2024 day-ahead market prices — official REN Data Hub API

**Primary source:** REN (Redes Energéticas Nacionais), official Data Hub electricity market price API; REN labels these market prices as sourced from OMIE. This notebook replaces the previous GitHub community archive with **REN's officially documented API**. It collects hourly Day-Ahead PT prices for all 366 days of 2024, explicitly checks completeness, and only then writes annual stats.

API documentation: https://datahub.ren.pt/pt/instrucoes-api/

REN daily endpoint: https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily?culture=pt-PT&date=2025-01-25

Direct OMIE public archive (alternative official source): https://www.omie.es/en/file-access-list?dir=+Day-ahead+market+hourly+price+in+Portugal&parents=%2FDay-ahead+Market%2F1.+Prices&realdir=marginalpdbcpt

**Execution status:** Notebook code provided; 2024 annual download and validation have NOT been completed within this chat due DNS failures in the execution runtime. No invented observations.

In [ ]:
from pathlib import Path
import datetime as dt
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

YEAR = 2024
BASE = 'https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily'
MONTHLY = 'https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesMonthly'
OUT = Path('data')
OUT.mkdir(exist_ok=True)
session = requests.Session()

In [ ]:
def get_json(url, params):
    error = None
    for attempt in range(4):
        try:
            response = session.get(url, params=params, timeout=30)
            response.raise_for_status()
            result = response.json()
            if not isinstance(result, dict):
                raise ValueError('Expected JSON object')
            return result
        except (requests.RequestException, ValueError) as exc:
            error = exc
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f'Failed official REN request {url} {params}: {error}')

def extract_pt_prices(payload, date):
    # Official REN endpoint returns {'xAxis': {...}, 'series': [{'name':'PT','data':[...]} ,...]}
    pt_series = [s for s in payload.get('series', []) if isinstance(s, dict) and s.get('name') == 'PT']
    if len(pt_series) != 1:
        raise ValueError(f'{date}: expected exactly one PT series, got {len(pt_series)}')
    values = pt_series[0].get('data')
    if not isinstance(values, list) or len(values) not in (23, 24, 25):
        raise ValueError(f'{date}: unexpected hourly length {len(values) if isinstance(values,list) else None}')
    prices = pd.to_numeric(pd.Series(values), errors='coerce')
    if prices.isna().any():
        raise ValueError(f'{date}: missing/non-numeric prices')
    return [{'date': date, 'hour_ordinal': i+1, 'price_eur_mwh': float(p)} for i,p in enumerate(prices)]

In [ ]:
dates = pd.date_range(f'{YEAR}-01-01', f'{YEAR}-12-31', freq='D')
records, failures = [], {}
for day in dates:
    date = day.strftime('%Y-%m-%d')
    try:
        payload = get_json(BASE, {'culture': 'pt-PT', 'date': date})
        records.extend(extract_pt_prices(payload, date))
    except Exception as error:
        failures[date] = str(error)
    if day.day == 1 or day == dates[-1]:
        print(date, '| records:', len(records), '| failures:', len(failures))

df = pd.DataFrame(records, columns=['date','hour_ordinal','price_eur_mwh'])
if not df.empty:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['date','hour_ordinal']).reset_index(drop=True)
print('Received days:', df.date.nunique() if not df.empty else 0, 'of 366')
print('Failures sample:', dict(list(failures.items())[:8]))

In [ ]:
expected = pd.date_range('2024-01-01','2024-12-31',freq='D')
observed = pd.DatetimeIndex(df.date.unique()) if not df.empty else pd.DatetimeIndex([])
missing = expected.difference(observed)
lengths = df.groupby('date').size() if not df.empty else pd.Series(dtype='int64')
invalid_lengths = lengths[~lengths.isin([23,24,25])]
complete = (len(failures)==0 and len(missing)==0 and invalid_lengths.empty and len(observed)==366)
print('Complete official REN 2024 extraction:', complete)
print('Missing dates:', [d.strftime('%Y-%m-%d') for d in missing])
print('Invalid day-length counts:', invalid_lengths.to_dict())
if not complete:
    raise RuntimeError('2024 extraction incomplete: annual statistics and official CSV intentionally blocked')

output = OUT / 'portugal_day_ahead_2024_REN_hourly.csv'
df.to_csv(output, index=False)
print('Saved complete official REN dataset:', output, len(df), 'hours')

In [ ]:
p = df.price_eur_mwh
summary = pd.Series({'count': len(p), 'days': df.date.nunique(), 'mean_eur_mwh': p.mean(), 'median_eur_mwh': p.median(), 'std_eur_mwh': p.std(), 'min_eur_mwh': p.min(), 'p05_eur_mwh': p.quantile(.05), 'p25_eur_mwh': p.quantile(.25), 'p75_eur_mwh': p.quantile(.75), 'p95_eur_mwh': p.quantile(.95), 'max_eur_mwh': p.max(), 'negative_hours': int((p<0).sum()), 'zero_hours': int((p==0).sum())})
monthly = df.assign(month=df.date.dt.to_period('M')).groupby('month').price_eur_mwh.agg(['count','mean','median','std','min','max'])
print(summary.to_string())
print(monthly.to_string())
summary.to_csv(OUT/'portugal_day_ahead_2024_REN_stats.csv',header=True)
monthly.to_csv(OUT/'portugal_day_ahead_2024_REN_monthly_stats.csv')

In [ ]:
# Cross-check monthly official REN figures; do not silently assume equivalent weighting.
monthly_reference = []
for month in range(1,13):
    try:
        item = get_json(MONTHLY, {'culture':'pt-PT','year':2024,'month':f'{month:02d}'})
        # REN's monthly JSON has Portuguese month keys containing a PT object and 'Preço Médio'.
        candidates = [v['PT']['Preço Médio'] for v in item.values() if isinstance(v,dict) and isinstance(v.get('PT'),dict) and 'Preço Médio' in v['PT']]
        if len(candidates)!=1:
            raise ValueError('Cannot uniquely identify PT monthly official average')
        monthly_reference.append({'month': f'2024-{month:02d}', 'official_REN_mean': float(candidates[0])})
    except Exception as error:
        print(f'Official monthly average unavailable for {month:02d}: {error}')
check = monthly.reset_index().assign(month=lambda x:x.month.astype(str)).merge(pd.DataFrame(monthly_reference),on='month',how='left')
check['difference_eur_mwh'] = check['mean'] - check['official_REN_mean']
print(check[['month','mean','official_REN_mean','difference_eur_mwh']].to_string(index=False))

In [ ]:
daily = df.groupby('date').price_eur_mwh.mean()
fig,ax = plt.subplots(figsize=(12,4))
daily.plot(ax=ax,title='Portugal 2024 day-ahead — daily average (official REN API)')
ax.set_ylabel('EUR/MWh')
plt.show()
fig,ax = plt.subplots(figsize=(12,4))
monthly['mean'].plot.bar(ax=ax,title='Portugal 2024 day-ahead — monthly average (official REN API)')
ax.set_ylabel('EUR/MWh')
plt.show()

## Data scope / reproducibility

- **PT Day-Ahead only**. Intraday auctions, continuous trading, aFRR, mFRR and imbalance prices require other distinct datasets.
- The public REN API's response shape was checked against a documented sample for 2025; the full 2024 extraction is still unverified in this environment. Run the notebook where REN is reachable.
- Daylight-saving dates may contain 23/25 periods; `hour_ordinal` avoids ambiguous clock labels.
- Official REN monthly average prices may use an aggregation convention worth checking before treating differences as mistakes.
- Retain 2025 notebook unchanged; this file covers only 2024.